# 14 — Advanced Extensions

Optional challenges for participants who finish early or want to go deeper.

Each section is **self-contained** — pick any that interest you.

---

## Extension A — KV-Cache for Fast Inference

During generation we process the same prefix tokens at every step. KV-cache stores the key/value tensors from previous steps so we only compute the new token at each step — reducing inference cost from $O(n^2)$ to $O(n)$.

## Extension B — Scale to TinyStories

Replace the character tokeniser with BPE and train on the TinyStories dataset.

## Extension C — MoE vs Dense Baseline

Replace the FFN in the model with a MoE layer and compare training curves.

## Extension D — Multimodal: Text + Images

Add a PatchEmbedding encoder and feed image patches alongside text.

---

In [ ]:
import sys, os, math, time
sys.path.insert(0, os.path.abspath('../..'))
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Extension A — KV-Cache

Without cache: each new token requires a full forward pass over the entire sequence ($O(n)$ per token = $O(n^2)$ total).

With cache: store past K and V tensors; only compute the new token's Q and update K/V ($O(1)$ per token).

In [ ]:
from src.generation.cache import KVCache
from src.models.transformer import TransformerLM, TransformerConfig

cfg = TransformerConfig(
    vocab_size=64, d_model=64, n_heads=4, n_layers=2,
    d_ff=256, max_seq_len=128, dropout=0.0,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
model = TransformerLM(cfg).to(device)
model.eval()

# Benchmark: generation without vs with cache
prompt = torch.randint(0, cfg.vocab_size, (1, 8), device=device)
N_TOKENS = 50
N_RUNS   = 5

# Without cache — naive loop
def generate_naive(model, prompt, n=N_TOKENS):
    ids = prompt.clone()
    with torch.no_grad():
        for _ in range(n):
            logits = model(ids)[:, -1, :]
            next_t = logits.argmax(-1, keepdim=True)
            ids    = torch.cat([ids, next_t], dim=1)
    return ids

# Time naive
t0 = time.perf_counter()
for _ in range(N_RUNS): generate_naive(model, prompt)
naive_ms = (time.perf_counter() - t0) / N_RUNS * 1000

print(f'Naive generation ({N_TOKENS} tokens): {naive_ms:.1f} ms/run')

# KV-cache generation is exposed via the library's cache module
print('\nSee src/generation/cache.py for the KV-cache implementation.')
print('Challenge: integrate it into the generation loop and measure the speedup.')

## Extension B — TinyStories with BPE

Requires:
```bash
python scripts/download_data.py --dataset tinystories --size small
pip install sentencepiece
```

In [ ]:
DATA_PATH = '../../data/raw/tinystories_small.txt'

if os.path.exists(DATA_PATH):
    with open(DATA_PATH) as f:
        text = f.read()
    print(f'TinyStories loaded: {len(text):,} characters')

    try:
        from src.data.tokenizer import BPETokenizer
        tok = BPETokenizer()
        # Train on first 50k chars to be fast
        sentences = [s.strip() for s in text[:50000].split('.') if len(s.strip()) > 10]
        tok.train(sentences, vocab_size=2000, model_prefix='tinystories_tok')
        print(f'Trained BPE tokenizer: vocab_size={tok.vocab_size}')

        sample = "once upon a time"
        ids = tok.encode(sample)
        print(f'Sample encoding: "{sample}" → {ids} → "{tok.decode(ids)}"')
    except ImportError:
        print('sentencepiece not installed. Run: pip install sentencepiece')
else:
    print('TinyStories not found. Run the download script first.')
    print('Skipping this extension.')

## Extension C — MoE vs Dense Comparison

In [ ]:
from src.models.moe import MoEBlock
from src.models.transformer_block import DecoderBlock

# Build a simple stacked model using MoE blocks
class MoETransformer(nn.Module):
    """Minimal transformer using MoE FFN layers."""
    def __init__(self, vocab_size, d_model, n_heads, n_layers, num_experts=4, top_k=2):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            MoEBlock(d_model, n_heads, num_experts=num_experts, top_k=top_k)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        h   = self.embed(x)
        aux = 0.0
        for blk in self.blocks:
            h, a = blk(h)
            aux  = aux + a
        return self.head(self.norm(h)), aux / len(self.blocks)


VOCAB, D, H, L = 64, 64, 4, 2

dense_model = TransformerLM(TransformerConfig(
    vocab_size=VOCAB, d_model=D, n_heads=H, n_layers=L, d_ff=256,
    max_seq_len=48, dropout=0.1, ffn_type='standard', norm_type='layernorm',
    pos_encoding='sinusoidal'
)).to(device)

moe_model = MoETransformer(VOCAB, D, H, L, num_experts=4, top_k=2).to(device)

dense_params = sum(p.numel() for p in dense_model.parameters())
moe_params   = sum(p.numel() for p in moe_model.parameters())
print(f'Dense params: {dense_params:,}')
print(f'MoE params  : {moe_params:,}   ({moe_params/dense_params:.1f}x more params, same active compute)')

print('\nChallenge: train both on the same data and compare val loss curves.')

## Extension D — Multimodal: Text + Synthetic Images

In [ ]:
from src.models.multimodal import PatchEmbedding, VisionEncoder
from src.models.attention import MultiHeadAttention

d_model = 64

# Vision encoder
vision_enc = VisionEncoder(
    image_size=32, patch_size=8,
    d_model=d_model, n_heads=4, n_layers=2
).to(device)

# Language model
lm_cfg = TransformerConfig(
    vocab_size=64, d_model=d_model, n_heads=4, n_layers=2,
    d_ff=256, max_seq_len=48, dropout=0.0,
    ffn_type='standard', norm_type='layernorm', pos_encoding='sinusoidal',
)
lm = TransformerLM(lm_cfg).to(device)

# Cross-attention bridge (text queries image KV)
cross_attn = MultiHeadAttention(d_model, n_heads=4).to(device)

# Demonstrate multimodal forward pass
images = torch.randn(2, 3, 32, 32, device=device)
texts  = torch.randint(0, 64, (2, 10), device=device)

with torch.no_grad():
    visual_tokens = vision_enc(images)           # (2, 16, 64)
    text_embed    = lm.embedding(texts)          # (2, 10, 64)
    # Text queries attend to image keys/values
    fused, _      = cross_attn(text_embed, visual_tokens, visual_tokens)

print(f'Images        : {images.shape}')
print(f'Visual tokens : {visual_tokens.shape}')
print(f'Text embed    : {text_embed.shape}')
print(f'Fused output  : {fused.shape}')

print('\nChallenge: build an image-captioning dataset (MNIST digits + label strings)')
print('and train the model to describe what it sees.')

## You're Done!

Thank you for completing the **Building a Gemini-Level Model from Scratch** workshop.

### What you built:
- Scaled dot-product attention & multi-head attention
- Positional encodings (sinusoidal, learned, RoPE)
- Feed-forward networks (standard & SwiGLU)
- Complete transformer blocks (Pre-LN, RMSNorm)
- Efficient attention (sliding window, linear)
- Mixture of Experts
- Multimodal fusion
- BPE tokenisation
- Full training pipeline with cosine LR schedule
- Text generation (greedy, temperature, top-k, top-p)

### Next steps:
- Read the original papers (see `docs/paper_references.md`)
- Fine-tune on domain-specific data
- Explore RLHF / preference learning
- Study inference optimisation (quantisation, speculative decoding)